In [2]:
import tensorflow as tf
import numpy as np

2026-03-24 14:38:09.383087: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Now there code of already prepared model

In [3]:

target_lichess_classes = [
    'exposedKing',
    'sacrifice',
    'hangingPiece',
    'fork',
    'captureTheDefender',
    'pin',
    'quietMove',
    'intermezzo',
    'deflection'
]
additional_target_classes = [
    'planlessGame'
]
num_classes = len(target_lichess_classes)+len(additional_target_classes)+1 #No class is not class but it must be given to loss too

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Bidirectional
#import keras
import numpy as np

#@keras.utils.register_keras_serializable()
class CNNLSTM(tf.keras.Model):
    """
    Модель CNN-LSTM для классификации последовательностей позиций.
    Принимает на вход позиции и оценки, возвращает вероятности классов.
    """
    def __init__(self, CNN=None, n_lstm_blocks=32, n_lstm_layers=2, bidirectional=True, window_length=5):
        self.window_length = window_length
        self.threshold = 0.5
        super(CNNLSTM, self).__init__()
        self.n_classes = len(target_lichess_classes) + len(additional_target_classes)
        path = 'models/tf_model_19x256.keras'

        if CNN is None:
            self.CNN = tf.keras.models.load_model(path)
        else:
            self.CNN = CNN
        
        
        
        
        self.lstm = tf.keras.models.Sequential([
            Bidirectional(tf.keras.layers.LSTM(n_lstm_blocks, return_sequences=True, name='LSTM1')),
            Bidirectional(tf.keras.layers.LSTM(n_lstm_blocks), name='LSTM2')
        ], name='LSTM')


        self.binary_classifier_head = tf.keras.models.Sequential([
            tf.keras.layers.Dense(units=16, activation='relu', name='binary_head_dense1'),
            tf.keras.layers.Dense(units=2, activation='softmax', name='binary_head_dense2')
        ], name='binary_classifier_head')

        self.multiclass_head = tf.keras.models.Sequential([
            tf.keras.layers.Dense(units=32, activation='relu', name='multiclass_head_dense1'),
            tf.keras.layers.Dense(units=16, activation='relu', name='multiclass_head_dense2'),
            tf.keras.layers.Dense(units=self.n_classes, activation='softmax', name='multiclass_head_dense3')
        ], name='multiclass_classifier_head')

        self.CNN.build(input_shape=(None, 8, 8, 112))
        self.lstm.build(input_shape=(None, 5, 24))
        self.binary_classifier_head.build(input_shape=(None, n_lstm_blocks*2))
        self.multiclass_head.build(input_shape=(None, n_lstm_blocks*2))


    def _process_CNN(self, inputs):
        """
        Обрабатывает батч позиций через CNN.
        Преобразует 5D тензор (batch, frames, H, W, D) в 3D (batch, frames, features).
        Использует векторизацию через reshape для ускорения.
        """
        inputs = tf.convert_to_tensor(inputs)
        if len(inputs.shape) == 4:
            inputs = tf.expand_dims(inputs, axis=0)

        shape = tf.shape(inputs)
        batch_size, n_frames = shape[0], shape[1]
        
        # Объединяем батч и время для пакетной обработки CNN
        inputs_reshaped = tf.reshape(inputs, [-1, shape[2], shape[3], shape[4]])
        cnn_out = self.CNN(inputs_reshaped)
        feature_dim = tf.shape(cnn_out)[-1]

        # Возвращаем размерность времени обратно
        return tf.reshape(cnn_out, [batch_size, n_frames, feature_dim])

    def _prepare_data(self, vects, evals):
        '''
           Prepares vectors of CNN output and evaluations for rnn
           Methods implemented there:
            - Batching
            - Setting length of input data to window length - clipping/padding
            - Extending evals along 1 dimension - repeating every scalar 8 times
            Takes - vects after CNN (tf.tensor), evals (unbatched or batched) (numpy array)
            Returns - prepared vects and evals tf tensors
        '''
        if len(evals.shape)==1:
            evals = np.expand_dims(evals, axis=0) #batch_size, n_frames


        if evals.shape[1]<self.window_length:
            evals = np.pad(evals, [[0,0], [0, 5-evals.shape[1]]])

        else:
            evals = evals[:, :self.window_length]
        #batch_size, 5

        evals = np.expand_dims(evals, axis=2)
        evals = tf.convert_to_tensor(evals)
        evals = tf.tile(evals, [1,1,8])
        #batch_size, 5, 8 : every scalar 8 times at axis 1
        assert len(evals.shape)==3, f"Eval's shape is {evals.shape}"
        # Паддинг до фиксированной длины 5
        if vects.shape[1] < 5:
            pad_frames = 5 - vects.shape[1]
            paddings = tf.constant([[0, 0], [0, pad_frames], [0, 0]])
            vects = tf.pad(vects, paddings)
            

        mask = tf.constant([1.0, 1.0, 0.1, 1.0, 0.1], dtype=tf.float32)
        mask = tf.reshape(mask, [1, 5, 1])
        vects = vects * mask
        return vects, evals

    def call(self, inputs):
        """
        Прямой проход модели.
        inputs: список [positions, evals]
        positions: np.array формы (batch, frames, H, W, D)
        evals: np.array формы (batch, frames,)
        Возвращает словарь с тензорами вероятностей для обучения.
        """
        positions, evals = inputs
        vects = self._process_CNN(positions)

        
        vects, evals = self._prepare_data(vects, evals)
        

        
        rnn_input = tf.concat([vects, evals], axis=2)

        after_rnn = self.lstm(rnn_input)

        binary_probas = self.binary_classifier_head(after_rnn)
        multiclass_probas = self.multiclass_head(after_rnn)

        return {
            'binary': binary_probas,
            'multiclass': multiclass_probas
        }

    def predict_decision(self, inputs):
        """
        Метод инференса для получения финального предсказания.
        Применяет пороговую логику к бинарному выходу.
        Возвращает -1 если бинарная вероятность ниже порога, иначе индекс класса.
        """
        outputs = self.call(inputs)
        binary_probas = outputs['binary']
        multiclass_probas = outputs['multiclass']

        # Векторизованное условие: если proba[1] < threshold -> -1, иначе argmax
        predictions = tf.where(
            binary_probas[:, 1] < self.threshold,
            tf.constant(-1, shape=[tf.shape(binary_probas)[0]]),
            tf.argmax(multiclass_probas, axis=1)
        )
        return predictions
    
    

model = CNNLSTM()

2026-03-24 14:38:13.575341: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
model([np.random.rand(5,8,8,112), np.random.rand(5)])
#model.summary()

{'binary': <tf.Tensor: shape=(1, 2), dtype=float32, numpy=array([[0.49995524, 0.50004476]], dtype=float32)>,
 'multiclass': <tf.Tensor: shape=(1, 10), dtype=float32, numpy=
 array([[0.09751281, 0.10235521, 0.10516807, 0.10845155, 0.10339288,
         0.094657  , 0.09430738, 0.09765462, 0.09881135, 0.09768919]],
       dtype=float32)>}